# ChainTranslate
English-to-French translation of Ethereum text

**Suraj Shuban · 230953492 · ICT4403 Assignment 3**

This notebook loads the public project, reproduces its analysis and can run all four model experiments. The published `results/` contains completed measurements for all four models, with 128 test translations and 48 temperature samples. LLaMA has the highest measured BLEU and chrF. New experiments are saved separately from the published outputs.

## 1 Prepare the project
In Colab, select a GPU runtime before running the experiment cells. A fresh runtime is recommended for dependency installation.

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo_url = "https://github.com/themisogynist/chaintranslate-web3-mt.git"
if not Path("chaintranslate/experiment.py").exists():
    project = Path.cwd() / "chaintranslate-web3-mt"
    if not project.exists():
        subprocess.run(["git", "clone", repo_url, str(project)], check=True)
    os.chdir(project)
assert Path("chaintranslate/experiment.py").exists()
print("Project:", Path.cwd())

In [ ]:
# Colab's preinstalled torchvision can be incompatible with the pinned torch.
# This benchmark uses text models only and does not require torchvision.
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("google.colab") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
# Use a fresh process, as each experiment does; no weights are downloaded here.
probe = subprocess.run(
    [sys.executable, "-c", "from transformers import MT5ForConditionalGeneration, LlamaForCausalLM, Gemma3ForCausalLM; import peft; print('All pretrained model imports passed')"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(probe.stdout)
if probe.returncode:
    raise RuntimeError("Dependency import check failed; inspect the error above before training.")


## 2 Inspect the fixed dataset and published measurements
Parallel entries share localization keys; files and exact English sources are disjoint across training, validation and test. Rebuilding the corpus is optional because the pinned splits are included.

In [ ]:
import json
from IPython.display import display, Markdown, Image
print(json.loads(Path("data/summary.json").read_text()))
display(Markdown(Path("RESULTS.md").read_text()))
display(Image(filename="results/scores.png"))
# Optional reconstruction from the pinned source:
# subprocess.run([sys.executable, "-m", "chaintranslate.data", "--out", "data"], check=True)

## 3 Authenticate for official gated models
Approved access is required for [LLaMA](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) and [Gemma](https://huggingface.co/google/gemma-3-270m-it). Authenticate through the secure widget. Credentials must not appear in notebook source, saved outputs or Git. Authentication alone does not grant model access.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import hf_hub_download
from chaintranslate.experiment import IDS, REVISIONS
# Check both gates before starting any expensive training.
for model in ["llama", "gemma"]:
    hf_hub_download(IDS[model], "config.json", revision=REVISIONS[model])
    print(model, "access verified")

## 4 Train and generate on the same test set
Experiments run sequentially in one hardware session. The basic Transformer is trained from scratch, mT5 is LoRA fine-tuned, and the official LLaMA and Gemma models use the same three training demonstrations. All use greedy decoding for primary scores and the same three sampling temperatures for the exploratory analysis. This cell may take substantial time on CPU. A fresh session creates a separate output directory. Retrying this cell in the same session reuses its directory and skips models with all three completed output files.

In [ ]:
from datetime import datetime, timezone
# Keep the same run directory when retrying a failed cell in this session.
if "run_dir" not in globals():
    run_dir = Path("runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir.mkdir(parents=True, exist_ok=True)
print("Results directory:", run_dir, flush=True)
for model in ["basic", "mt5", "llama", "gemma"]:
    expected = [run_dir / f"{model}_{suffix}.json"
                for suffix in ["metrics", "translations", "temperatures"]]
    if all(path.exists() for path in expected):
        print(model, "already completed; skipping", flush=True)
        continue
    print("Running", model, flush=True)
    command = [sys.executable, "-u", "-m", "chaintranslate.experiment",
               "--model", model, "--out", str(run_dir)]
    # Forward output explicitly so Colab shows the actual cause of a failure.
    with subprocess.Popen(command, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f"{model} failed with exit code {returncode}; see its error above. Results remain in {run_dir}.")
print("Results:", run_dir)


## 5 Validate and compare
Reporting checks alignment, recomputes the scores and requires all temperature samples for every reported model. The status file identifies missing models. For a failed run, retain its output directory and rerun the affected model using `--out` with that same directory before reporting.

In [ ]:
subprocess.run([sys.executable, "-m", "chaintranslate.report", "--out", str(run_dir), "--report", str(run_dir / "RESULTS.md")], check=True)
display(Markdown((run_dir / "RESULTS.md").read_text()))
print((run_dir / "status.json").read_text())
display(Image(filename=str(run_dir / "scores.png")))
display(Image(filename=str(run_dir / "learning_curves.png")))

## 6 Examine actual translations and ethical risks
Inspect the same IDs across models for omissions, unsupported additions, terminology, quantities, warnings and repetition. Record counterexamples as well as successful outputs. The automated flags are review candidates, not a safety or fairness certification.

In [ ]:
print((run_dir / "translations.txt").read_text()[:12000])
print((run_dir / "temperature_metrics.json").read_text())
print((run_dir / "review_candidates.json").read_text())

## 7 Export measured outputs
The export includes all new scores, histories and translations, the exact data splits and the source used for the run. It excludes credentials, caches and model checkpoints.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
export_path = Path("runs") / (run_dir.name + "_measured_results.zip")
with ZipFile(export_path, "w", ZIP_DEFLATED) as archive:
    for path in run_dir.rglob("*"):
        if path.is_file() and "checkpoints" not in path.parts:
            archive.write(path, Path("results") / path.relative_to(run_dir))
    for folder in ["data", "chaintranslate"]:
        for path in Path(folder).rglob("*"):
            if path.is_file() and "__pycache__" not in path.parts:
                archive.write(path, path)
    archive.write("requirements.txt")
print(export_path)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(export_path))